# NMODL to BrainCell Walkthrough

This notebook shows the full inspectable path for the focused BrainCell conversion flow:

1. `.mod -> AST`
2. `AST -> RawBlocks`
3. `RawBlocks + AST -> CanonicalBlocks`
4. `CanonicalBlocks -> one_ion_hh_ohmic IR`
5. `IR + Jinja -> generated Python class`

CLI counterparts:

- `steps/inspect_ast/cli.py`
- `steps/inspect_ir/cli.py`
- `steps/render/cli.py`
- `examples/generate_braincell.py`

Current registered combination:

- `canonical_default -> one_ion_hh_ohmic -> braincell_one_ion_hh_ohmic`


In [1]:
from pathlib import Path
import json
import sys

repo_root = Path.cwd()
if not (repo_root / "steps").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from steps import ast_to_json
from steps import build_canonical_blocks
from steps import collect_block_counts
from steps import build_one_ion_hh_ohmic_ir
from steps import extract_raw_blocks
from steps import parse_program
from steps import render_one_ion_hh_ohmic

mod_file = repo_root / "mod_files" / "kv.mod"
program = parse_program(mod_file)
raw_blocks = extract_raw_blocks(program)
canonical_blocks = build_canonical_blocks(program)
braincell_ir = build_one_ion_hh_ohmic_ir(canonical_blocks, mod_file, raw_blocks=raw_blocks)
rendered_python = render_one_ion_hh_ohmic(braincell_ir)

print(mod_file)


/home/swl/braincell/examples/convert_mod/nmodl/mod_files/kv.mod


--No graphics will be displayed.
/home/swl/anaconda3/envs/braincell/lib/python3.10/site-packages/nmodl/dsl.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import *


## Step 1: AST preview


In [2]:
ast_payload = json.loads(ast_to_json(program))
print(program.get_node_type_name())
print(json.dumps(ast_payload, indent=2)[:1500])


Program
{
  "Program": [
    {
      "Model": [
        {
          "String": [
            {
              "name": " Simplified Kv example for BrainCell template generation\r"
            }
          ]
        }
      ]
    },
    {
      "NeuronBlock": [
        {
          "StatementBlock": [
            {
              "Suffix": [
                {
                  "Name": [
                    {
                      "String": [
                        {
                          "name": "SUFFIX"
                        }
                      ]
                    }
                  ]
                },
                {
                  "Name": [
                    {
                      "String": [
                        {
                          "name": "Kv"
                        }
                      ]
                    }
                  ]
                }
              ]
            },
            {
              "Useion": [
                {
               

## Step 2: Raw blocks


In [3]:
print([name for name, values in raw_blocks.items() if values])
print(json.dumps(raw_blocks, indent=2)[:1500])


['TITLE', 'NEURON', 'UNITS', 'PARAMETER', 'ASSIGNED', 'STATE', 'INITIAL', 'BREAKPOINT', 'DERIVATIVE', 'FUNCTION', 'PROCEDURE']
{
  "TITLE": [
    "TITLE Simplified Kv example for BrainCell template generation"
  ],
  "COMMENT": [],
  "NEURON": [
    "NEURON {\n    SUFFIX Kv\n    USEION k READ ek WRITE ik\n    RANGE gbar, i, v12, q\n}"
  ],
  "UNITS": [
    "UNITS {\n    (mA) = (milliamp)\n    (mV) = (millivolt)\n    (S) = (siemens)\n}"
  ],
  "PARAMETER": [
    "PARAMETER {\n    gbar = 0.0 (S/cm2)\n    Ra = 0.02 (/mV/ms)\n    Rb = 0.006 (/mV/ms)\n    v12 = 25 (mV)\n    q = 9 (mV)\n}"
  ],
  "ASSIGNED": [
    "ASSIGNED {\n    v (mV)\n    i (mA/cm2)\n    ik (mA/cm2)\n    ek (mV)\n    ninf\n    ntau (ms)\n    tadj\n}"
  ],
  "STATE": [
    "STATE {\n    n\n}"
  ],
  "INITIAL": [
    "INITIAL {\n    rates(v)\n    n = ninf\n}"
  ],
  "BREAKPOINT": [
    "BREAKPOINT {\n    SOLVE states METHOD cnexp\n    i = gbar*n*(v-ek)\n    ik = i\n}"
  ],
  "DERIVATIVE": [
    "DERIVATIVE states {\n    ra

## Step 3: Canonical blocks


In [4]:
print(json.dumps(canonical_blocks["NEURON"], indent=2))
print(json.dumps(canonical_blocks["BREAKPOINT"], indent=2))


{
  "suffix": "Kv",
  "useion": [
    {
      "ion": "k",
      "read": [
        "ek"
      ],
      "write": [
        "ik"
      ]
    }
  ],
  "range": [
    "gbar",
    "i",
    "v12",
    "q"
  ],
  "global": [],
  "nonspecific_current": []
}
{
  "solve_stmt": {
    "target": "states",
    "method": "cnexp"
  },
  "func_calls": [],
  "statements": [
    {
      "assigned_var": "i",
      "expression": "gbar*n*(v-ek)"
    },
    {
      "assigned_var": "ik",
      "expression": "i"
    }
  ],
  "other_statements": []
}


## Step 4: BrainCell IR


In [5]:
print(json.dumps(braincell_ir, indent=2)[:2000])


{
  "source_file": "/home/swl/braincell/examples/convert_mod/nmodl/mod_files/kv.mod",
  "title": "Simplified Kv example for BrainCell template generation",
  "mechanism_name": "Kv",
  "class_name": "IK_Kv",
  "template_name": "one_ion_hh_ohmic.py",
  "supported": true,
  "rejection_reasons": [],
  "manual_fix_required": [],
  "unsupported_blocks": {},
  "base_class_name": "PotassiumChannel",
  "ion_name": "k",
  "ion_arg_name": "K",
  "g_max_param": {
    "target_name": "g_max",
    "safe_name": "g_max",
    "source_name": "gbar",
    "default_expression": "0.0 * (u.S / (u.cm ** 2))",
    "unit": "S/cm2"
  },
  "v_shift_param": {
    "target_name": "V_sh",
    "safe_name": "V_sh",
    "default_expression": "0. * u.mV"
  },
  "temperature_param": {
    "target_name": "temp",
    "safe_name": "temp",
    "default_expression": "u.celsius2kelvin(23)"
  },
  "tref_expression": "u.celsius2kelvin(23)",
  "extra_parameters": [
    {
      "source_name": "Ra",
      "safe_name": "Ra",
      "de

## Step 5: Rendered Python


In [6]:
print("\n".join(rendered_python.splitlines()))


"""Generated one-ion HH ohmic channel from /home/swl/braincell/examples/convert_mod/nmodl/mod_files/kv.mod."""



import braintools
import brainunit as u

from braincell.quad import DiffEqState
from braincell.channel import PotassiumChannel


class IK_Kv(PotassiumChannel):
    __module__ = "braincell.channel"

    def __init__(
        self,
        size,
        g_max=0.0 * (u.S / (u.cm ** 2)),
        V_sh=0. * u.mV,
        temp=u.celsius2kelvin(23),
        Ra=0.02 * (1 / u.mV / u.ms),
        Rb=0.006 * (1 / u.mV / u.ms),
        q=9 * (u.mV),
        v12=25 * (u.mV),
        name=None,
    ):
        super().__init__(size=size, name=name)

        self.g_max = braintools.init.param(g_max, self.varshape, allow_none=False)
        self.V_sh = braintools.init.param(V_sh, self.varshape, allow_none=False)
        self.temp = temp
        self.Ra = braintools.init.param(Ra, self.varshape, allow_none=False)
        self.Rb = braintools.init.param(Rb, self.varshape, allow_none=False)
   

## Step 6: Save The Rendered Result

This writes the final rendered BrainCell channel, IR, and core earlier step outputs into one artifact directory under `examples/artifacts/`.


In [7]:
artifact_dir = repo_root / "examples" / "artifacts" / mod_file.stem
artifact_dir.mkdir(parents=True, exist_ok=True)

ast_payload = json.loads(ast_to_json(program))
block_counts = dict(collect_block_counts(program))
(artifact_dir / "rendered_channel.py").write_text(rendered_python, encoding="utf-8")
(artifact_dir / "braincell_ir.json").write_text(json.dumps(braincell_ir, indent=2), encoding="utf-8")
(artifact_dir / "ast.json").write_text(json.dumps(ast_payload, indent=2), encoding="utf-8")
(artifact_dir / "block_counts.json").write_text(json.dumps(block_counts, indent=2), encoding="utf-8")
(artifact_dir / "raw_blocks.json").write_text(json.dumps(raw_blocks, indent=2), encoding="utf-8")
(artifact_dir / "canonical_blocks.json").write_text(json.dumps(canonical_blocks, indent=2), encoding="utf-8")

metadata = {
    "source_file": str(mod_file),
    "saved_files": [
        "rendered_channel.py",
        "braincell_ir.json",
        "ast.json",
        "block_counts.json",
        "raw_blocks.json",
        "canonical_blocks.json",
    ],
}
(artifact_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(artifact_dir)


/home/swl/braincell/examples/convert_mod/nmodl/examples/artifacts/kv


To try a rejection case, replace `kv.mod` with `hh.mod` and rerun the notebook.
